In [ ]:
"""
Herbarium Processor Notebook
========================

This notebook is designed to process herbarium specimen images and generate labels using a prompt-based approach.
It includes steps for image processing, prompt configuration, and generating specimen labels.
It is structured to be run in a Jupyter notebook environment.

Configuration: select what images to sample from.

You should edit this section each time you run the code to:
  * select the images you want to process
  * set the path to the output CSV file.
"""

from utils.notebook_setup import setup_project

setup_project()

image_path = "data/img/bucket"
output_csv_path = "tmp/an_output.csv"

Notebook root set to /Users/danielleward/Documents/Projects/parsely_studio


In [27]:
"""
Configuration: setting up the prompts and system instructions for Gemini inference.

You do NOT need to edit this section.
"""

from herbarium_processor.core.inference.prompt_factory import (
    create_prompt_builder_from_yaml,
)

system_instructions_path = "prompts/ocr_system_instructions_no_citations.md"
prompt_builder = create_prompt_builder_from_yaml("prompts/configs/default_prompt.yaml")

print("Prompt:")
print(prompt_builder.generate_debug_prompt())

Prompt:
You are an AI assistant trained to extract structured data from herbarium specimen label images.

You will be shown 3 label images and corresponding OCR blocks as examples. These examples include the correct structured JSON output. After that, you will be shown a fourth image (with OCR blocks) that you must process.

Use both the image and the OCR output to extract only the following 21 fields as a valid JSON object using `snake_case` keys:

* `catalogNumber`
* `datasetName`
* `scientificName`
* `locality`
* `municipality`
* `county`
* `stateProvince`
* `country`
* `verbatimLatitude`
* `verbatimLongitude`
* `verbatimElevation`
* `habitat`
* `verbatimEventDate`
* `eventDate`
* `recordedBy`
* `recordNumber`
* `identifiedBy`
* `dateIdentified`
* `otherCatalogNumbers`
* `associatedTaxa`
* `comment`


Return `null` for any field that is not clearly present or legible. Do **not** infer or fabricate any information. Do **not** return extra fields.

---
**Example 1**

Image:
<|image_0|

In [23]:
"""
Step 1: Image processing.

Take the .heic images from the phone and convert them to .jpg format.
This step also includes resizing the images to a standard size and doing any necessary preprocessing.

You do NOT need to edit this section.
"""

from herbarium_processor.core.image.image_utils import process_directory_no_auto_rotate

process_directory_no_auto_rotate(image_path)

[.DS_Store] Failed to process image: cannot identify image file '/Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/.DS_Store'
[cs_16625_25.jpg] Original size: (980, 787)
[cs_16625_25.jpg] No resize needed.
[cs_16625_25.jpg] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/cs_16625_25.jpg
[cs_16625_23.jpg] Original size: (980, 787)
[cs_16625_23.jpg] No resize needed.
[cs_16625_23.jpg] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/cs_16625_23.jpg
[UC46732_2.JPG] Original size: (990, 1000)
[UC46732_2.JPG] No resize needed.
[UC46732_2.JPG] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/UC46732_2.JPG
[cs_16625_15.jpg] Original size: (949, 1000)
[cs_16625_15.jpg] No resize needed.
[cs_16625_15.jpg] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/cs_16625_15.jpg
[cs_16625_14.jpg] Origin

In [24]:
"""
Step 2: Calling the OCR AI model.
This step will call OCR and save the output to .json files in the tmp/ directory.

It will then build specimen labels based on the OCR output.
You do NOT need to edit this section.
"""

import os

from herbarium_processor.core.ocr.ocr_client import OcrClient

ocr = OcrClient()

if image_path:
    for filename in os.listdir(image_path):
        path = os.path.join(image_path, filename)
        if filename.lower().endswith(".jpg") or filename.lower().endswith(".jpeg"):
            print(f"Processing {path}")
            ocr.extract_text_json(path)

Processing data/img/bucket/cs_16625_25.jpg
Saved visualized image with bounding boxes and sources to /Users/danielleward/Documents/Projects/parsely_studio/tmp/ocr_bounding_cs_16625_25.jpg
[{'id': '1.1.1', 'text': 'Herbarium bryologicum Hjalmar Möller', 'bounding_box': [{'x': 237, 'y': 169}, {'x': 700, 'y': 169}, {'x': 700, 'y': 193}, {'x': 237, 'y': 193}], 'average_confidence': 0.9812338775}, {'id': '1.1.2', 'text': 'Grimmia unieder Hork', 'bounding_box': [{'x': 165, 'y': 229}, {'x': 695, 'y': 229}, {'x': 695, 'y': 285}, {'x': 165, 'y': 285}], 'average_confidence': 0.7800799666666668}, {'id': '1.1.3', 'text': 'Dahlen , Amat på Söder Trollholm', 'bounding_box': [{'x': 127, 'y': 380}, {'x': 805, 'y': 380}, {'x': 805, 'y': 433}, {'x': 127, 'y': 433}], 'average_confidence': 0.5336712583333333}, {'id': '1.1.4', 'text': 'Vinern .', 'bounding_box': [{'x': 680, 'y': 433}, {'x': 806, 'y': 433}, {'x': 806, 'y': 454}, {'x': 680, 'y': 454}], 'average_confidence': 0.461798135}, {'id': '1.1.5', 'tex

In [25]:
"""
Step 3: Building the samples for the prompt based on the OCR output.

This section will automatically generate the list of specimen labels
based on the sampled paths defined above.

You do NOT need to edit this section.
"""

from pathlib import Path

from herbarium_processor.core.types.specimen_label import SpecimenLabel

target_examples = []

if image_path:
    for filename in os.listdir(image_path):
        if filename.startswith(".DS"):
            continue
        path = Path(os.path.join(image_path, filename))
        if path.suffix.lower() not in {".jpg", ".jpeg"}:
            continue
        ocr_file = Path("tmp") / f"ocr_ai_input_{path.stem}.json"
        if not ocr_file.exists():
            print(f"Skipping {filename}, OCR file not found: {ocr_file}")
            continue
        label = SpecimenLabel(
            id=path.stem,
            img_path=str(path),
            ocr_path=f"tmp/ocr_ai_input_{path.stem}.json",
        )
        target_examples.append(label)
        print("Saving specimen label with the following info:" + str(label))

Saving specimen label with the following info:SpecimenLabel(id='cs_16625_25', img_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/cs_16625_25.jpg'), ocr_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/tmp/ocr_ai_input_cs_16625_25.json'))
Saving specimen label with the following info:SpecimenLabel(id='cs_16625_23', img_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/cs_16625_23.jpg'), ocr_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/tmp/ocr_ai_input_cs_16625_23.json'))
Saving specimen label with the following info:SpecimenLabel(id='UC46732_2', img_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/UC46732_2.JPG'), ocr_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/tmp/ocr_ai_input_UC46732_2.json'))
Saving specimen label with the following info:SpecimenLabel(id='cs_16625_15', img_path=PosixPath('/Users/daniell

In [26]:
"""
Step 4: Calling the AI model

You do NOT need to edit this section.

"""

from herbarium_processor.core.inference.label_extraction_batch_runner import (
    LabelExtractionBatchRunner,
)

runner = LabelExtractionBatchRunner(
    output_csv_path=output_csv_path,
    system_instructions_path=system_instructions_path,
    prompt_builder=prompt_builder,
    targets=target_examples,
)
await runner.run_async()

[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
Saved outp